# parser_dev trial notebook

This notebook is for direct trial-and-error calls against the current codebase.
It does not patch production code; it just exercises the functions we expect to edit next.

Current direction for the upcoming implementation pass:
- keep edits minimal and close to existing yarp-again style
- always generate `_atom_info` for each yarpecule
- keep `atom_index` and `atom_map` separate
- preserve partial user mappings and never overwrite existing user-provided maps
- use call-site `strict=False` fallback from YARP parsing to RDKit parsing
- inspect the full RDKit output with `verbose=True` in this notebook only
- keep all generated development files under `dev/`


In [ ]:
from pathlib import Path
from pprint import pprint
import sys
import types

try:
    import openbabel  # type: ignore
except ModuleNotFoundError:
    class _DummyErrorLog:
        def SetOutputLevel(self, *_args, **_kwargs):
            return None

    openbabel_stub = types.ModuleType('openbabel')
    openbabel_stub.pybel = types.SimpleNamespace()
    openbabel_stub.openbabel = types.SimpleNamespace(obErrorLog=_DummyErrorLog(), obError=0)
    sys.modules['openbabel'] = openbabel_stub

from yarp.yarpecule.graph.smiles import smiles2adjmat
from yarp.yarpecule.input_parsers import xyz_from_smiles, xyz_parse, xyz_q_parse, mol_parse
from yarp.yarpecule.yarpecule import yarpecule

REPO = Path.cwd()
TEST_DIR = REPO / 'test'
DEV_DIR = REPO / 'dev'
GENERATED_DIR = DEV_DIR / 'generated'
GENERATED_DIR.mkdir(exist_ok=True)

XYZ_HAA = TEST_DIR / 'molecules' / 'haa.xyz'
MOL_ETHANOL = TEST_DIR / 'molecules' / 'ethanol.mol'

SMILES_CASES = {
    'haa_unmapped': 'O=CCO',
    'haa_mapped': '[C:0]([C:1]([O:7][H:11])([H:13])[H:14])(=[O:6])[H:12]',
    'aromatic_mapped': '[c:0]1([H:15])[c:4]([O:9][H:13])[c:5]([H:20])[c:1]([H:16])[o:6]1',
}


In [ ]:
def summarize_smiles_case(name, smiles):
    adj, bem, atom_info = smiles2adjmat(smiles)
    print(f'CASE: {name}')
    print(f'  atoms={len(atom_info)} charge={int(sum(info[1] for info in atom_info))}')
    print(f'  adj shape={adj.shape} bem shape={bem.shape}')
    print('  first atom_info entries:')
    pprint(atom_info[:min(4, len(atom_info))])
    print()

for name, smiles in SMILES_CASES.items():
    summarize_smiles_case(name, smiles)


## Target behavior for next implementation pass

- `_atom_info` should exist for SMILES, XYZ, and MOL construction paths.
- `_atom_info` should separate `atom_index` from `atom_map`.
- canonical reorder may change `atom_index`, but should not change `atom_map`.
- partial user mappings should be preserved exactly.
- `strict=False` should allow a warning-backed fallback from YARP parsing to RDKit parsing.
- `verbose=True` should print the full RDKit mol representation before and after map injection.
- XYZ export should place RDKit canonical smiles in the comment line.


In [ ]:
TARGET_ASSERTIONS = {
    'smiles_atom_info_present': True,
    'xyz_atom_info_present': True,
    'mol_atom_info_present': True,
    'atom_index_and_atom_map_are_separate': True,
    'partial_user_maps_are_preserved': True,
    'atom_map_survives_reorder': True,
    'strict_false_warns_then_falls_back': True,
    'verbose_prints_full_rdkit_before_after_mapping': True,
    'xyz_comment_has_rdkit_canonical_smiles': True,
}
print(TARGET_ASSERTIONS)


In [ ]:
for mode in ('yarp', 'rdkit'):
    elements, geo, adj_mat, q = xyz_from_smiles(SMILES_CASES['haa_unmapped'], mode=mode)
    print(f'mode={mode} atoms={len(elements)} q={q} geo_shape={geo.shape} adj_shape={adj_mat.shape}')


In [ ]:
elements_xyz, geo_xyz = xyz_parse(str(XYZ_HAA))
q_xyz = xyz_q_parse(str(XYZ_HAA))
elements_mol, geo_mol, adj_mol, q_mol = mol_parse(str(MOL_ETHANOL))

print('XYZ parse:')
print('  elements:', elements_xyz)
print('  geo shape:', geo_xyz.shape)
print('  q:', q_xyz)
print()
print('MOL parse:')
print('  first elements:', elements_mol[:5])
print('  geo shape:', geo_mol.shape)
print('  adj shape:', adj_mol.shape)
print('  q:', q_mol)


In [ ]:
# These calls are intended to become verbose=True once the production code supports it.
# Keep them explicit here so the notebook already reflects the target workflow.

y_smi_unmapped = yarpecule(SMILES_CASES['haa_unmapped'], mode='yarp', canon=False)
y_smi_mapped = yarpecule(SMILES_CASES['haa_mapped'], mode='yarp', canon=False)
y_xyz = yarpecule(str(XYZ_HAA), canon=False)

# Target call shape after implementation:
# y_smi_unmapped.get_smiles(verbose=True)
# y_smi_mapped.get_smiles(verbose=True)
# y_xyz.get_smiles(verbose=True)

y_smi_unmapped.get_smiles()
y_smi_mapped.get_smiles()
y_xyz.get_smiles()

print('yarpecule from unmapped smiles:')
print('  canon_smi:', y_smi_unmapped.canon_smi)
print('  map_smi:', y_smi_unmapped.map_smi)
print('  _atom_info:', y_smi_unmapped._atom_info)
print()
print('yarpecule from mapped smiles:')
print('  canon_smi:', y_smi_mapped.canon_smi)
print('  map_smi:', y_smi_mapped.map_smi)
print('  _atom_info:', y_smi_mapped._atom_info)
print()
print('yarpecule from xyz:')
print('  canon_smi:', y_xyz.canon_smi)
print('  map_smi:', y_xyz.map_smi)
print('  _atom_info:', y_xyz._atom_info)


In [ ]:
xyz_path = GENERATED_DIR / 'haa_unmapped.xyz'
mol_path = GENERATED_DIR / 'haa_unmapped.mol'
y_smi_unmapped.export_geometry(str(xyz_path), format='xyz')
y_smi_unmapped.export_geometry(str(mol_path), format='mol')
print('FILES FOR haa_unmapped')
print('  xyz path:', xyz_path)
print('  mol path:', mol_path)
print('  xyz contents:')
print(xyz_path.read_text())
print('  mol contents:')
print(mol_path.read_text())
print()

xyz_path = GENERATED_DIR / 'haa_mapped.xyz'
mol_path = GENERATED_DIR / 'haa_mapped.mol'
y_smi_mapped.export_geometry(str(xyz_path), format='xyz')
y_smi_mapped.export_geometry(str(mol_path), format='mol')
print('FILES FOR haa_mapped')
print('  xyz path:', xyz_path)
print('  mol path:', mol_path)
print('  xyz contents:')
print(xyz_path.read_text())
print('  mol contents:')
print(mol_path.read_text())
print()

xyz_path = GENERATED_DIR / 'haa_xyz.xyz'
mol_path = GENERATED_DIR / 'haa_xyz.mol'
y_xyz.export_geometry(str(xyz_path), format='xyz')
y_xyz.export_geometry(str(mol_path), format='mol')
print('FILES FOR haa_xyz')
print('  xyz path:', xyz_path)
print('  mol path:', mol_path)
print('  xyz contents:')
print(xyz_path.read_text())
print('  mol contents:')
print(mol_path.read_text())


## What to inspect next

- `smiles2adjmat()` still returns list-based `atom_info` and has no call-site fallback handling yet.
- `xyz_from_smiles()` still returns a 4-tuple, so `_atom_info` is not propagated upstream.
- `yarpecule` built from XYZ/MOL currently leaves `_atom_info` as `None`.
- `get_smiles()` currently writes atom maps from RDKit atom indices, not preserved user maps.
- the current notebook uses the final target call shape in comments, but production code does not support `verbose=True` yet.
- `xyz_write()` does not yet place RDKit canonical smiles in the XYZ comment line.
- the atom-mass dictionary does not yet live in `yarp/util/constants.py`.
